In [11]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q pyspark findspark

In [12]:
import os
import findspark

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

findspark.init()

In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Internship") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.1


## Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

### Driver
- The Driver is the main program that controls the Spark application.
- It creates the SparkSession, schedules tasks, and coordinates execution.
- It converts user code into execution plans and distributes work to executors.

### Cluster Manager
- The Cluster Manager allocates resources to Spark applications.
- It manages CPU and memory across worker nodes.
- Examples include Standalone Cluster Manager, YARN, Mesos, and Kubernetes.

### Executor
- Executors are worker processes that execute tasks assigned by the Driver.
- They store data in memory or disk and return results to the Driver.
- Each application has one or more executors running on worker nodes.

## Q2. How does Spark’s Lazy Evaluation strategy improve performance when processing large datasets?

Spark uses Lazy Evaluation, meaning transformations are not executed immediately. Instead, Spark records all transformations and builds a Directed Acyclic Graph (DAG). Execution starts only when an action such as show(), count(), or collect() is called.

Benefits:
- Reduces unnecessary computations.
- Optimizes execution using DAG.
- Minimizes disk I/O.
- Combines multiple transformations into an efficient execution plan.
- Improves performance for large datasets.

## Q3. Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [19]:
df = spark.read \
     .option("header", True) \
     .option("inferSchema", True) \
     .csv("/content/final_employees.csv")

df.show()

+----------+-------------+---+------+--------------+------------------+------+----------+---------+-----------+--------------------+----------+-----------+-----+----------+-----------+
|EmployeeID|         Name|Age|Gender|    Department|       Designation|Salary|Experience|     City|JoiningDate|               Email|product_id|   category|price|base_price|final_price|
+----------+-------------+---+------+--------------+------------------+------+----------+---------+-----------+--------------------+----------+-----------+-----+----------+-----------+
|      1001|Ananya Sharma| 45|Female|            HR| Marketing Manager| 44144|         4|    Noida| 2024-03-25|ananya.sharma1@ex...|      2001|Electronics|49999|   49999.0|   58998.82|
|      1002| Divya Sharma| 22|  Male|            HR| Marketing Manager| 68118|        20|Bangalore| 2024-08-02|divya.sharma2@exa...|      2002|  Furniture|29999|   29999.0|   35398.82|
|      1003|  Divya Singh| 36|Female|Administration| Software Engineer| 847

## Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

| CSV | Parquet |
|------|----------|
| Row-based storage | Columnar storage |
| Larger file size | Compressed storage |
| Slower read performance | Faster read performance |
| No schema support | Supports schema |
| Reads entire rows | Reads only required columns |

Parquet provides better performance because it stores data column-wise, supports compression, and enables optimizations such as Predicate Pushdown, reducing disk I/O and improving query execution speed.

## Q5. Given a DataFrame df, write a query to select the columns product_id and price where the category is "Electronics".

In [20]:
df.filter(df.category == "Electronics") \
.select("product_id", "price") \
.show()

+----------+-----+
|product_id|price|
+----------+-----+
|      2001|49999|
|      2014|49999|
|      2015| 1499|
|      2018|  999|
|      2020|15999|
|      2022|24999|
|      2026| 4999|
|      2031|29999|
|      2039| 7999|
|      2046| 4999|
|      2052|15999|
|      2053|49999|
|      2058| 4999|
|      2061|49999|
|      2062|15999|
|      2072|  999|
|      2073| 1499|
|      2083| 2499|
|      2084| 9999|
|      2099|24999|
+----------+-----+



# Q6. Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [21]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("old_name", "new_name") \
       .withColumn("price", col("price").cast("double"))

df.printSchema()

root
 |-- EmployeeID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Designation: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- JoiningDate: date (nullable = true)
 |-- Email: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- final_price: double (nullable = true)



# Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Spark records all transformations in a Lineage Graph (DAG) instead of storing intermediate data after every operation. If a worker node fails, Spark uses the DAG to recompute only the lost partitions from the original data rather than reprocessing the entire dataset. This provides efficient fault tolerance while minimizing recovery time and resource usage.

# Q8. Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [22]:
from pyspark.sql import Row

orders = [
    Row(order_id=1, status="Completed", amount=1500),
    Row(order_id=2, status="Pending", amount=800),
    Row(order_id=3, status="Completed", amount=2500),
    Row(order_id=4, status="Completed", amount=700)
]

df_orders = spark.createDataFrame(orders)

df_orders.filter(
    (df_orders.status == "Completed") &
    (df_orders.amount > 1000)
).show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|  1500|
|       3|Completed|  2500|
+--------+---------+------+



# Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown is an optimization technique used with Parquet files where Spark pushes filter conditions to the storage layer. Only the rows that satisfy the filter are read from disk, reducing disk I/O, memory usage, and execution time. Since unnecessary data is never loaded into memory, queries become much faster and more efficient.

# Q10. Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [23]:
from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

df.show()

+----------+-------------+---+------+--------------+------------------+------+----------+---------+-----------+--------------------+----------+-----------+-------+----------+------------------+
|EmployeeID|         Name|Age|Gender|    Department|       Designation|Salary|Experience|     City|JoiningDate|               Email|product_id|   category|  price|base_price|       final_price|
+----------+-------------+---+------+--------------+------------------+------+----------+---------+-----------+--------------------+----------+-----------+-------+----------+------------------+
|      1001|Ananya Sharma| 45|Female|            HR| Marketing Manager| 44144|         4|    Noida| 2024-03-25|ananya.sharma1@ex...|      2001|Electronics|49999.0|   49999.0|          58998.82|
|      1002| Divya Sharma| 22|  Male|            HR| Marketing Manager| 68118|        20|Bangalore| 2024-08-02|divya.sharma2@exa...|      2002|  Furniture|29999.0|   29999.0|          35398.82|
|      1003|  Divya Singh| 36|

# Q11. What is the difference between Transformations and Actions? Provide two examples of each


| Transformations | Actions |
|-----------------|---------|
| Create a new DataFrame/RDD without executing immediately. | Trigger the execution of all pending transformations. |
| Follow Lazy Evaluation. | Execute immediately and produce a result. |
| Return a new DataFrame/RDD. | Return a value or write data to storage. |
| Used to build the execution plan (DAG). | Used to display, save, or collect the final output. |
| **Examples:** `filter()`, `select()` | **Examples:** `show()`, `collect()` |

# Q12. Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [25]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/content/final_employees.csv")

In [26]:
df.write.mode("overwrite").parquet("/content/employee_parquet")

In [27]:
df = spark.read.parquet("/content/employee_parquet")

df.filter(df.EmployeeID.isNotNull()) \
  .write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("/content/output_csv")

In [28]:
spark.read.csv("/content/output_csv", header=True).show()

+----------+-------------+---+------+--------------+------------------+------+----------+---------+-----------+--------------------+----------+-----------+-----+----------+-----------+
|EmployeeID|         Name|Age|Gender|    Department|       Designation|Salary|Experience|     City|JoiningDate|               Email|product_id|   category|price|base_price|final_price|
+----------+-------------+---+------+--------------+------------------+------+----------+---------+-----------+--------------------+----------+-----------+-----+----------+-----------+
|      1001|Ananya Sharma| 45|Female|            HR| Marketing Manager| 44144|         4|    Noida| 2024-03-25|ananya.sharma1@ex...|      2001|Electronics|49999|   49999.0|   58998.82|
|      1002| Divya Sharma| 22|  Male|            HR| Marketing Manager| 68118|        20|Bangalore| 2024-08-02|divya.sharma2@exa...|      2002|  Furniture|29999|   29999.0|   35398.82|
|      1003|  Divya Singh| 36|Female|Administration| Software Engineer| 847

# Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

| Client Mode | Cluster Mode |
|--------------|--------------|
| Driver program runs on the client machine. | Driver program runs inside the cluster. |
| Client must remain connected during execution. | Client can disconnect after submitting the job. |
| Mainly used for development and testing. | Mainly used for production environments. |

# Q14. Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [29]:
from pyspark.sql import Row

tasks = [
    Row(id=1, region="North", priority="Low"),
    Row(id=2, region="South", priority="High"),
    Row(id=3, region="North", priority="High"),
    Row(id=4, region="East", priority="Medium")
]

df_tasks = spark.createDataFrame(tasks)

df_tasks.filter(
    (df_tasks.region == "North") |
    (df_tasks.priority == "High")
).show()

+---+------+--------+
| id|region|priority|
+---+------+--------+
|  1| North|     Low|
|  2| South|    High|
|  3| North|    High|
+---+------+--------+



# Q15. When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

The show(5) method displays only the first five rows of a dataset, making it memory-efficient and safe for large datasets. In contrast, collect() retrieves the entire dataset to the driver node, which can consume excessive memory and may crash the application when working with multi-terabyte datasets. Therefore, show(5) is the recommended choice for previewing large datasets.